# Experiments

## Instance-level visual retrieval

Use this notebook to configure experiments, call your implementations, plot results, and record your analysis. Implement the algorithms in the package `instance_retrieval/`; use this notebook as your experiment log and writeup.

Let we start with the task table in the README. Complete the 10 TODOs in P1–P6 before running the full benchmark. Experimental results and answers are intentionally left blank. If you encounter `NotImplementedError`, implement the corresponding function in the package.


## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import platform
import sys
import subprocess
# Set these before importing numerical libraries.
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from instance_retrieval.utils import seed_everything
from instance_retrieval.data import RevisitedDataset, DATASETS

## Basic settings

In [ ]:
ROOT = Path.cwd().resolve()
assert (ROOT / "instance_retrieval").is_dir(), "Open writeup.ipynb from the project root"
DATA_PATH = ROOT / "data"
CACHE_PATH = ROOT / "cache" / "coursework"
RESULT_PATH = ROOT / "results" / "coursework"
SEED = 214 # Anyway, I like this number :p
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
seed_everything(SEED)

def check(selector):
    """Run public tests in this notebook's interpreter; failures are not suppressed."""
    subprocess.run([sys.executable, "-m", "pytest", "-q", "-k", selector], cwd=ROOT, check=True)

print({"device": DEVICE, "seed": SEED, "result_path": str(RESULT_PATH)})

## Data and evaluation protocol

Data loading, query bounding-box cropping, and the official evaluation code are provided. Inspect a full query image and its region of interest (ROI), and verify that database images follow the official `imlist` order.


In [ ]:
datasets = {name: RevisitedDataset(DATA_PATH, name) for name in DATASETS}
display(pd.DataFrame([{"dataset": name, "images": len(ds.imlist), "queries": len(ds.qimlist)}
                      for name, ds in datasets.items()]))
ds = datasets["roxford5k"]
from PIL import Image
with Image.open(ds.query_paths[0]) as image:
    original = image.convert("RGB")
query_image = ds.image(0, query=True)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(original)
axes[0].set_title("Full query")
axes[1].imshow(query_image)
axes[1].set_title("Bounding-box crop")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### P1 — Protocol adapter

Implement `evaluation.protocol_ground_truth` to convert easy/hard/junk annotations into the ok/junk format expected by the official `compute_map`. Follow the protocol definitions in the course PDF. The provided `evaluate` function validates rankings and calls the official evaluation code.

**Check:** `pytest -q -k protocol`. The example below uses a ranking that you can evaluate by hand. Verify the protocol on this example before evaluating real data.


In [ ]:
from instance_retrieval.evaluation import evaluate
check("protocol")
toy_ranks = np.array([[0], [1], [2], [3]])
toy_ground_truth = [{"easy": [1], "hard": [3], "junk": [0]}]
toy_metrics = evaluate(toy_ranks, toy_ground_truth)
display({p: {"mAP": m["mAP"], "mP@10": m["mP@10"]} for p, m in toy_metrics.items()})

**Written question 1.** Which results are ignored under the Medium (M) and Hard (H) protocols in the example above? Why does removing junk results change AP? Why is classification accuracy not a substitute for mAP in instance retrieval?

**Your answer:** TODO


## Representation and similarity

### P2 — Exact cosine retrieval

Implement `search.cosine_search` to return complete rankings and a similarity matrix, both of shape `[database_size, num_queries]`. Handle unnormalized vectors, zero vectors, and ties.

**Check:** `pytest -q -k search`.


In [ ]:
from instance_retrieval.search import cosine_search
check("search")
database_vectors = np.array([[2., 0.], [0., 5.], [1., 0.]])
query_vectors = np.array([[7., 0.], [0., 3.]])
ranks, scores = cosine_search(database_vectors, query_vectors)
display(scores)
display(ranks)

### P3 — CLS and patch pooling

Pretrained weight loading, DINOv2/CLIP forward inference, and image preprocessing are provided. Implement `deep.pool_dino_tokens` to return either the CLS token or positive GeM pooling over patch tokens only. See the function docstring for the input and output contracts.

**Check:** `pytest -q -k pool`. These tests require neither a GPU nor model downloads. After completing the implementation, run the real-image example below.


In [ ]:
from instance_retrieval.deep import DeepExtractor
from instance_retrieval.experiments import CONFIGS, run_deep
check("pool")
DINO_METHOD = "dino_s14_518_cls"
CLIP_METHOD = "clip_b32_224"
extractor = DeepExtractor(CONFIGS[DINO_METHOD], device=DEVICE)
query_feature = extractor.encode(query_image)
print("dimension:", query_feature.shape, "L2 norm:", np.linalg.norm(query_feature))
del extractor
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Deep feature experiments

Compare both pretrained model families on the same queries, and compare input resolutions of 224/518 and CLS/GeM pooling. The cell below runs the full benchmark and may take several minutes. First make sure the small tests for P1–P3 pass. Cache keys depend on the current implementation and configuration, so matching cached features can be reused.

You may change the parameters, but document every change and retain comparable control configurations.


In [ ]:
DEEP_METHODS = [
    "dino_s14_224_cls", "dino_s14_518_cls",
    "dino_s14_224_gem", "dino_s14_518_gem", CLIP_METHOD,
]
run_deep(ROOT, methods=DEEP_METHODS, datasets=DATASETS, device=DEVICE)

**Written question 2.** Compare the effects of input resolution and pooling strategy. What changes when GeM clamps patch-token activations that contain both positive and negative values? Which experimental results support your conclusions?

**Your answer:** TODO


## Local features and visual vocabulary

### P4 — SIFT, RootSIFT, sampling and k-means

Complete four functions in `local.py`: `extract_sift`, `rootsift`, `sample_descriptors`, and `train_vocabulary`.

Keep at most the 1,000 strongest keypoints per image and store raw descriptors as uint8. Train the vocabulary using approximately one million descriptors sampled uniformly from database images only. You may use OpenCV SIFT and Faiss k-means. Check shapes, norms, and empty-image behavior on small inputs before processing the full databases.

**Check:** `pytest -q -k 'sift or sample or vocabulary'`.


In [ ]:
from instance_retrieval.local import (
    extract_sift, rootsift, LocalStore, sample_descriptors, train_vocabulary, encode_vlad
)
check("sift or sample or vocabulary")
query_xy, query_descriptors = extract_sift(query_image)
print("xy:", query_xy.shape, "descriptors:", query_descriptors.shape, query_descriptors.dtype)
root_descriptors = rootsift(query_descriptors)
print("RootSIFT row norms:", np.linalg.norm(root_descriptors[:5], axis=1))

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(query_image)
ax.scatter(query_xy[:100, 0], query_xy[:100, 1], s=8, facecolors="none", edgecolors="yellow")
ax.axis("off")
plt.show()

**Written question 3.** Is sampling the same fixed number of descriptors from every image equivalent to sampling uniformly across all descriptors? How does your implementation control memory usage? Record the actual sample count and random seed.

**Your answer:** TODO


### P5 — VLAD encoding

Implement `local.encode_vlad`. Assign descriptors to their nearest centers and aggregate residuals, then apply intra-normalization, a signed square root, and global L2 normalization. The tests cover empty images, empty clusters, and a small example that can be computed by hand.

**Check:** `pytest -q -k vlad`. The cell below checks the encoding interface using only a few database images; it is not a full benchmark. The full vocabulary is trained later by `run_local`.


In [ ]:
check("vlad")
small_pool = [rootsift(extract_sift(ds.image(i))[1]) for i in range(5)]
small_sample = np.concatenate(small_pool, axis=0)
centers_demo = train_vocabulary(small_sample, k=64, seed=SEED, iterations=10)
query_vlad = encode_vlad(query_descriptors, centers_demo)
print("VLAD dimension:", query_vlad.shape, "L2 norm:", np.linalg.norm(query_vlad))

## Geometric verification

### P6 — RANSAC and Top-N reranking

Complete `geometry.geometric_inliers` and `geometry.rerank`. You may use OpenCV matching and homography RANSAC; implement match filtering, inlier counting, and reranking yourself. Only reorder the Top-N results, preserve the original order for ties, and leave the remaining results unchanged.

**Check:** `pytest -q -k 'geometry or rerank'`. The known planar transformation below is a small sanity check.


In [ ]:
from instance_retrieval.geometry import geometric_inliers
check("geometry or rerank")
rng = np.random.default_rng(SEED)
xy = rng.uniform(0, 200, (20, 2)).astype(np.float32)
desc = rng.integers(0, 200, (20, 128), dtype=np.uint8)
print("Inliers on a known transformation:",
      geometric_inliers((xy, desc), (xy * 1.1 + 10, desc)))

### Traditional pipeline experiments

Run all combinations of K=64/256 with and without RANSAC. Run the full public test suite first. The provided `run_local` handles caching, experiment orchestration, and result saving, and calls your implementations throughout the traditional pipeline. Explain your choice of Top-N.


In [ ]:
from instance_retrieval.experiments import run_local, tables
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT, check=True)
VOCABULARY_SIZES = (64, 256)
TOP_N = 100
WORKERS = 8
run_local(ROOT, sizes=VOCABULARY_SIZES, top_n=TOP_N, workers=WORKERS)

**Written question 4.** Which queries improve with RANSAC, and which become worse? Can reranking recover positives outside the initial Top-N? Explain the benefits and costs in terms of dimensionality, bytes per global vector, local-feature storage, and runtime.

**Your answer:** TODO


## Results and writeup

### E1 — Main results and baseline reproduction

Report mAP and mP@10 under M/H, storage requirements, and runtime alongside a reference row from the DINOv2 paper. Reference values are provided; measured values must come from running your own implementation.

If any gap exceeds five percentage points, first check cropping, evaluation protocols, model settings, and preprocessing, then discuss the discrepancy using ablation evidence.


In [ ]:
results = tables(ROOT)
display(results.round(1))
display(results.pivot_table(index=["dataset", "method"], columns="protocol",
                            values=["mAP", "mP@10"]).round(1))
display(pd.read_csv(RESULT_PATH / "baseline_comparison.csv").round(1))

### E2 — Vocabulary size × geometric reranking

Compare the traditional pipeline configurations using tables and plots. Distinguish feature extraction time, vector search time, and geometric verification time.


In [ ]:
traditional = results[results.method.str.startswith("rootsift")]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, name in zip(axes, DATASETS):
    traditional[traditional.dataset == name].pivot(
        index="method", columns="protocol", values="mAP"
    ).plot.bar(ax=ax)
    ax.set_title(name)
    ax.set_ylabel("mAP (%)")
plt.tight_layout()
plt.show()

**Written question 5.** Summarize your main results and baseline reproduction. Support each conclusion with a specific table or experimental configuration. Distinguish established findings from explanations that remain untested.

**Your answer:** TODO


### E3 — Semantic gap and failure analysis

Compare the Top-10 results for the same query across methods. Green indicates positives, red indicates incorrect results, and gray indicates results ignored by the evaluation protocol. Select at least three failed queries, explain the causes using visual evidence, and summarize the error distributions for the two model families. Do not infer instance identity from similar filenames alone.


In [ ]:
from instance_retrieval.visualization import load_rankings, top10_figure
DATASET = "roxford5k"
QUERY_INDEX = 0
PROTOCOL = "hard"
COMPARE_METHODS = [DINO_METHOD, CLIP_METHOD, "rootsift_vlad256_ransac"]
rankings, similarities = load_rankings(ROOT, DATASET, COMPARE_METHODS)
top10_figure(datasets[DATASET], QUERY_INDEX, rankings, PROTOCOL, similarities)
plt.show()

In [ ]:
from instance_retrieval.analysis import prepare_failures, failure_summary
annotations = prepare_failures(ROOT)
display(annotations.head())
print("Edit the category and explanation columns in", RESULT_PATH / "failure_annotations.csv")
# Rerun after completing the annotations:
summary, coverage = failure_summary(ROOT)
display(coverage)
display(summary)

**Written question 6.** Is the gap between the two model families the same under M and H? Which failure categories are most common? Why does category-level semantic similarity not guarantee instance identity? Discuss how model size, preprocessing, and sampling choices limit your conclusions.

**Your answer:** TODO


## Demo

After completing the implementations and experiments, use the provided Gradio interface to demonstrate image uploads, Top-10 retrieval, and performance metrics. The cell below starts the service; you may skip this section when running experiments only. Alternatively, run `python -m instance_retrieval.cli demo` in a terminal. Add your own demo screenshot to the README.


In [ ]:
from instance_retrieval.demo import RetrievalDemo
demo = RetrievalDemo(ROOT, dataset="roxford5k", method=DINO_METHOD, device=DEVICE)
app = demo.build()
app.launch(server_name="127.0.0.1", server_port=17861,
           share=False, inline=True, prevent_thread_lock=True)